💡 **Environment:** `clamp-analyses` (R kernel)


# Supplementary Figure 7: drug-disease prediction across top-k thresholds and precision-recall

One 180 × 85 mm page with all text at least 5 pt. Inputs come from `drug_disease_method_comparison` and
`drug_disease_topk_comparison` (`workflow/rules/archs4_drug_diseases.smk`), which compare the methods listed in
`comparison_methods` of `workflow/config/archs4.yaml` against the gene-based approach.

- **a**: AUROC of the maximum over 49 GTEx tissues when each trait keeps its top k modules, for every k averaged in
  Figure 4; the dashed line is the gene-based approach.
- **b**: AUPRC per GTEx tissue with scores averaged over the top-k thresholds; diamonds mark the maximum over
  tissues, the dashed line the positive rate, and brackets two-sided paired Wilcoxon tests (BH q) of
  ARCHS4 CLAMPfull against each other method.


In [ ]:
source(here::here("scripts/panels/main_figure.R"))
set.seed(1)
dd_root <- here("output/03_model_biology/02_archs4/02_drug_diseases_canonical/method_comparison")
dirs <- main_figure_dirs("supp7")
ord <- names(DRUG_DISEASE_METHOD_COLORS)
cols <- DRUG_DISEASE_METHOD_COLORS
k_dirs <- list.files(dd_root, pattern = "^top[0-9]+$", full.names = TRUE)
ks <- sort(as.integer(sub("^top", "", basename(k_dirs))))
stopifnot(length(ks) > 1)
ka <- rbindlist(lapply(ks, function(k) fread(file.path(dd_root, paste0("top", k), "max_aggregate_reference.csv"))[, k := k]))
ka[, `:=`(method = factor(method, levels = ord), x = match(k, ks))]
tissue <- fread(file.path(dd_root, "data", "per_tissue_metrics.csv"))[, method := factor(method, levels = ord)]
agg <- fread(file.path(dd_root, "data", "max_aggregate_reference.csv"))[, method := factor(method, levels = ord)]
stopifnot(!anyNA(ka$method), !anyNA(tissue$method), uniqueN(tissue$tissue) == 49, "auprc" %in% names(tissue))
gene_max <- ka[method == "Gene-based", auroc][1]
base_rate <- tissue$n_positive[1] / tissue$n_pairs[1]
wide <- dcast(tissue, tissue ~ method, value.var = "auprc")
tests <- rbindlist(lapply(ord[-1], function(m) data.table(group1 = ord[1], group2 = m,
  median_difference = median(wide[[ord[1]]] - wide[[m]]),
  p_value = wilcox.test(wide[[ord[1]]], wide[[m]], paired = TRUE, alternative = "two.sided", exact = FALSE)$p.value)))
tests[, q_value_bh := p.adjust(p_value, "BH")]
tests[, `:=`(xstart = 1L, xend = match(group2, ord))]
setorder(tests, xend)
fwrite(ka[order(method, k), .(method, k, auroc, auprc)], file.path(dirs$source, "a_topk_max_auroc.csv"))
fwrite(tissue, file.path(dirs$source, "b_per_tissue_auprc.csv"))
fwrite(agg, file.path(dirs$source, "b_max_aggregate.csv"))
fwrite(tests[, .(group1, group2, median_difference, p_value, q_value_bh)], file.path(dirs$source, "b_paired_tests.csv"))


## Panel a: top-k sensitivity


In [ ]:
fmt_q <- function(q) {
  if (q >= 0.001) return(sprintf("italic(q) == %.3f", q))
  parts <- strsplit(formatC(q, format = "e", digits = 1), "e")[[1]]
  sprintf("italic(q) == \"%s\" %%*%% 10^{%d}", trimws(parts[1]), as.integer(parts[2]))
}
tag <- function(p, l) p + labs(tag = l) + theme(plot.tag = element_text(size = 8, face = "bold"), plot.tag.position = c(0.005, 0.985))
topk_lines <- ggplot(ka[method != "Gene-based"], aes(x, auroc, colour = method)) +
  geom_hline(aes(yintercept = gene_max, linetype = "Gene-based"), colour = cols[["Gene-based"]], linewidth = 0.3) +
  geom_line(linewidth = 0.4) +
  geom_point(size = 1.1) +
  scale_x_continuous(breaks = seq_along(ks), labels = ks, expand = expansion(add = 0.3)) +
  scale_colour_manual(values = cols, name = NULL) +
  scale_linetype_manual(values = c("Gene-based" = "dashed"), name = NULL) +
  labs(x = "Modules kept per trait (top-k)", y = "AUROC, max over tissues") +
  theme_nm() +
  theme(legend.position = "bottom", legend.direction = "vertical", legend.box = "horizontal",
        legend.key.width = unit(3, "mm"), legend.key.height = unit(2.4, "mm")) +
  guides(colour = guide_legend(ncol = 2, order = 1, override.aes = list(shape = 16, size = 1.1)),
         linetype = guide_legend(order = 2))


## Panel b: AUPRC per tissue


In [ ]:
lo <- min(tissue$auprc)
step <- 0.009
lab_y <- merge(agg, tissue[, .(vmax = max(auprc)), by = method], by = "method")[, y := pmax(auprc, vmax) + 0.004]
top_y <- max(lab_y$y) + 0.004
auprc_violin <- ggplot(tissue, aes(method, auprc, fill = method)) +
  geom_hline(yintercept = base_rate, linetype = "dashed", colour = "grey60", linewidth = 0.25) +
  geom_violin(width = 0.7, trim = TRUE, scale = "width", colour = "black", linewidth = 0.22, alpha = 0.65, show.legend = FALSE) +
  geom_boxplot(width = 0.18, outlier.shape = NA, colour = "black", linewidth = 0.22, alpha = 0.85, show.legend = FALSE) +
  geom_jitter(width = 0.2, height = 0, size = 0.5, shape = 21, fill = "white", colour = "#333333", stroke = 0.15, alpha = 0.65) +
  geom_point(data = agg, aes(method, auprc, shape = "Maximum over\ntissues"), size = 1.05, fill = "white",
             colour = "black", stroke = 0.28, inherit.aes = FALSE) +
  geom_text(data = lab_y, aes(method, y, label = sprintf("%.3f", auprc)), vjust = 0, size = 5 / .pt, fontface = "bold", inherit.aes = FALSE) +
  geom_text(data = tissue[, .(med = median(auprc)), by = method], aes(method, lo - 0.006, label = sprintf("%.3f", med)),
            size = 5 / .pt, inherit.aes = FALSE) +
  scale_fill_manual(values = cols) +
  scale_shape_manual(values = c("Maximum over\ntissues" = 23), name = NULL) +
  labs(x = NULL, y = "AUPRC, mean over top-k\n(per GTEx tissue)") +
  coord_cartesian(ylim = c(lo - 0.01, top_y + 0.008 + step * (nrow(tests) + 0.8))) +
  theme_nm() +
  theme(legend.position = "right", legend.margin = margin(0, 0, 0, 0), axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1))
for (i in seq_len(nrow(tests))) {
  y <- top_y + 0.008 + step * i
  auprc_violin <- auprc_violin +
    annotate("segment", x = tests$xstart[i], xend = tests$xstart[i], y = y - 0.0025, yend = y, linewidth = 0.22) +
    annotate("segment", x = tests$xstart[i], xend = tests$xend[i], y = y, yend = y, linewidth = 0.22) +
    annotate("segment", x = tests$xend[i], xend = tests$xend[i], y = y - 0.0025, yend = y, linewidth = 0.22) +
    annotate("text", x = (tests$xstart[i] + tests$xend[i]) / 2, y = y + 0.002, label = fmt_q(tests$q_value_bh[i]),
             parse = TRUE, size = 5 / .pt, vjust = 0)
}


## Assemble and save


In [ ]:
supp <- plot_grid(tag(topk_lines, "a"), tag(auprc_violin, "b"), ncol = 2, rel_widths = c(0.95, 1.05))
save_main_figure(supp, "supp7", 180, 85)
options(repr.plot.width = 180 / 25.4, repr.plot.height = 85 / 25.4)
print(supp)
